# Tarea 1 – Parte 1: Recolección y limpieza de noticias

**Pregunta guía:** ¿se pueden convertir titulares de un diario peruano en un clasificador de noticias?

| Configuración | Valor |
| :--- | :--- |
| Fecha de descarga | **2026-10-18** (hora de Lima) |
| Opción elegida | **B – El Comercio** (350–400 noticias, ~90–100 por categoría) |
| URL base | `https://elcomercio.pe` |
| `robots.txt` | `https://elcomercio.pe/robots.txt` |
| Índices de sitemaps (declarados en `robots.txt`) | `https://elcomercio.pe/sitemap/{seccion}/?outputType=xml` |
| Sitemaps usados (URL y fecha de cada nota) | `https://elcomercio.pe/sitemap/web/{seccion}/?outputType=xml` con `seccion` = `politica`, `economia`, `deporte-total`, `tecnologia` |
| Titular, bajada y fecha | HTML de cada nota (`requests` + `BeautifulSoup`) |
| Categorías finales | `politica`, `economia`, `deportes`, `ciencia_tecnologia` |

**¿Por qué sitemaps + HTML y no solo RSS?**

- El Comercio tiene un RSS oficial por sección (`/arc/outboundfeeds/google-news-feed/category/{seccion}/`), pero solo cubre las últimas ~48 horas. El 18/10 traía 20 notas de política, 13 de economía, 86 de deportes y 15 de tecnología, así que no alcanza para ~90 por categoría.
- Los sitemaps de sección que publica `robots.txt` listan las 100 notas más recientes de cada sección, con su URL y su fecha. Con esa lista se visita cada nota **una sola vez** para leer el titular, la bajada y la fecha de publicación.
- Se descartó La República (opciones A y C) porque sus RSS y sitemaps por sección traen solo entre 6 y 21 notas, y su `robots.txt` prohíbe paginar las secciones (`Disallow: /*/pag/`).

**Buenas prácticas de scraping:** se revisa `robots.txt` antes de descargar, se usa un `User-Agent` descriptivo con enlace de contacto, hay una pausa de 1.5 s entre solicitudes y como máximo 1 reintento por URL. Todo lo descargado se respalda en `datos/raw/`; al reejecutar el notebook no se vuelve a pedir al servidor lo que ya está guardado.

In [ ]:
import json
import time
import urllib.robotparser
from datetime import date
from pathlib import Path
from urllib.parse import urlparse

import pandas as pd
import plotly.express as px
import plotly.io as pio
import requests
import truststore
from bs4 import BeautifulSoup

# Valida HTTPS con los certificados del sistema operativo. En Windows, el antivirus
# intercepta HTTPS con su propio certificado raíz, que el certifi de requests no conoce.
truststore.inject_into_ssl()

# Plotly: interactivo en Jupyter/VS Code + copia PNG para que GitHub muestre los gráficos
pio.renderers.default = "plotly_mimetype+png"
pio.renderers["png"].width = 950
pio.renderers["png"].height = 500
pd.set_option("display.max_colwidth", 110)

OPCION = "B"
MEDIO = "El Comercio"
BASE = "https://elcomercio.pe"
USER_AGENT = ("Mozilla/5.0 (compatible; TareaDiplomadoPython/1.0; uso academico; "
              "+https://github.com/IvettexMamani/diplomado-python-2026)")
PAUSA_SEG = 1.5      # entre 1 y 2 s entre solicitudes
MAX_REINTENTOS = 1   # como máximo 1 reintento por URL

# Secciones del diario cuyo sitemap se recorre
SITEMAPS = ["politica", "economia", "deporte-total", "tecnologia"]
ETIQUETAS = ["politica", "economia", "deportes", "ciencia_tecnologia"]

DATOS = Path("datos")
RAW = DATOS / "raw"
(RAW / "sitemaps").mkdir(parents=True, exist_ok=True)

print("Fecha de ejecución:", date.today().isoformat())
print(f"Opción {OPCION}: {MEDIO} ({BASE})")

## 1. Revisión de `robots.txt`
Se descarga `robots.txt` (y se guarda una copia en `datos/raw/`), se leen las reglas que aplican a cualquier robot (`User-agent: *`) y se comprueba con `urllib.robotparser` que el `User-Agent` del proyecto puede leer los sitemaps y las notas.

In [ ]:
sesion = requests.Session()
sesion.headers.update({"User-Agent": USER_AGENT, "Accept-Language": "es-PE,es;q=0.9"})
_ultima = [0.0]  # momento de la última solicitud


def descargar(url):
    # GET respetuoso: pausa mínima entre solicitudes y como máximo 1 reintento.
    r = None
    for intento in range(1 + MAX_REINTENTOS):
        espera = PAUSA_SEG - (time.time() - _ultima[0])
        if espera > 0:
            time.sleep(espera)
        try:
            r = sesion.get(url, timeout=25)
        except requests.RequestException as e:
            print(f"  error de red ({type(e).__name__}) en {url}")
            r = None
        _ultima[0] = time.time()
        if r is not None and r.status_code in (200, 404, 410):
            return r  # éxito, o la página no existe: no tiene sentido reintentar
    return r


archivo_robots = RAW / "robots_elcomercio.txt"
if archivo_robots.exists():
    texto_robots = archivo_robots.read_text(encoding="utf-8")
    print("robots.txt leído del respaldo en datos/raw/")
else:
    r = descargar(f"{BASE}/robots.txt")
    r.raise_for_status()
    texto_robots = r.text
    archivo_robots.write_text(texto_robots, encoding="utf-8")
    print("robots.txt descargado y guardado en datos/raw/")

rp = urllib.robotparser.RobotFileParser()
rp.parse(texto_robots.splitlines())

# Reglas del grupo "User-agent: *", que es el que aplica a nuestro User-Agent
reglas, en_grupo = [], False
for linea in texto_robots.splitlines():
    linea = linea.strip()
    if linea.lower().startswith("user-agent:"):
        en_grupo = linea.split(":", 1)[1].strip() == "*"
    elif en_grupo and linea.lower().startswith(("disallow:", "allow:")):
        reglas.append(linea)
sitemaps_declarados = [linea.split(":", 1)[1].strip() for linea in texto_robots.splitlines()
                       if linea.lower().startswith("sitemap:")]

print("Reglas para 'User-agent: *':", reglas)
print("Sitemaps declarados en robots.txt:", len(sitemaps_declarados))
print("Índices de las 4 secciones declarados:",
      all(f"{BASE}/sitemap/{s}/?outputType=xml" in sitemaps_declarados for s in SITEMAPS))

In [ ]:
pruebas = {
    "Índice de sitemaps de política": f"{BASE}/sitemap/politica/?outputType=xml",
    "Sitemap web de deportes": f"{BASE}/sitemap/web/deporte-total/?outputType=xml",
    "Nota de economía (ejemplo)": f"{BASE}/economia/mercados/ejemplo-noticia/",
    "Nota de tecnología (ejemplo)": f"{BASE}/tecnologia/actualidad/ejemplo-noticia/",
    "Buscador": f"{BASE}/buscar/ejemplo/",
    "Catálogo": f"{BASE}/catalogo/",
}
pd.DataFrame({"recurso": list(pruebas), "url": list(pruebas.values()),
              "permitido": [rp.can_fetch(USER_AGENT, u) for u in pruebas.values()]})

**Conclusión:** para cualquier robot, `robots.txt` solo prohíbe `/catalogo/`, `/buscar/` y el archivo histórico (`/archivo/todas/19*`, `/archivo/todas/200*`). El bloqueo total (`Disallow: /`) aplica solo a una lista de descargadores masivos (HTTrack, wget, WebZIP, etc.), y nuestro `User-Agent` no está en ella. Los sitemaps de sección y las páginas de las notas están permitidos.

**Fuente definida:** sitemaps oficiales por sección + HTML de cada nota.

## 2. Extracción

### 2.1 Sitemaps por sección
Para cada sección se abre su índice declarado en `robots.txt` y se toma el sitemap `web`, que lista las 100 notas más recientes. El XML crudo se guarda en `datos/raw/sitemaps/`.

In [ ]:
filas = []
for seccion in SITEMAPS:
    archivo = RAW / "sitemaps" / f"{seccion}.xml"
    if archivo.exists():
        xml, origen = archivo.read_text(encoding="utf-8"), "respaldo local"
    else:
        indice = f"{BASE}/sitemap/{seccion}/?outputType=xml"
        assert indice in sitemaps_declarados, f"{indice} no figura en robots.txt"
        r = descargar(indice)
        r.raise_for_status()
        hijos = [loc.get_text(strip=True) for loc in BeautifulSoup(r.text, "xml").find_all("loc")]
        url_web = next(u for u in hijos if "/sitemap/web/" in u)
        r = descargar(url_web)
        r.raise_for_status()
        xml, origen = r.text, f"descargado de {url_web}"
        archivo.write_text(xml, encoding="utf-8")
    urls = BeautifulSoup(xml, "xml").find_all("url")
    for u in urls:
        filas.append({"sitemap": seccion, "url": u.loc.get_text(strip=True),
                      "lastmod": u.lastmod.get_text(strip=True) if u.lastmod else None})
    print(f"{seccion:<14} {len(urls):>4} URLs  ({origen})")

df_urls = pd.DataFrame(filas)
print("Total de URLs en los 4 sitemaps:", len(df_urls))
df_urls.head()

### 2.2 Sección original y filtro previo
La `seccion_original` se toma de la ruta de la URL, tal como la organiza el diario (por ejemplo `economia/mercados` o `deporte-total/futbol-peruano`). Hay que tomarla de la URL y no del sitemap, porque un sitemap a veces incluye notas de otra sección.

Antes de visitar las páginas se descartan tres casos, para no hacer solicitudes innecesarias:

- **URLs repetidas** entre sitemaps.
- **Columnas de opinión** (`opinion/...`, `economia/opinion`). Son textos de autor, no noticias.
- **URLs prohibidas por `robots.txt`** (control de seguridad).

Cada descarte se anota con su motivo en la lista `descartes`, que se reporta en la sección 3.3.

In [ ]:
def seccion_desde_url(url):
    # Ruta de sección tal como la usa el diario; la última parte de la URL es el slug de la nota.
    partes = [p for p in urlparse(url).path.strip("/").split("/") if p]
    return "/".join(partes[:-1][:2])


descartes = []  # (url, motivo) de cada noticia descartada en cualquier etapa


def descartar(df, mascara, motivo):
    descartes.extend((u, motivo) for u in df.loc[mascara, "url"])
    print(f"{motivo:<55} -> {int(mascara.sum()):>3} descartadas")
    return df[~mascara].copy()


df_urls["seccion_original"] = df_urls["url"].map(seccion_desde_url)

candidatas = descartar(df_urls, df_urls.duplicated("url"), "URL repetida en dos sitemaps")
candidatas = descartar(candidatas, candidatas["seccion_original"].str.contains(r"(?:^|/)opinion"),
                       "Columna de opinión (no es noticia)")
candidatas = descartar(candidatas, ~candidatas["url"].map(lambda u: rp.can_fetch(USER_AGENT, u)),
                       "URL prohibida por robots.txt")
print("URLs candidatas a descargar:", len(candidatas))
candidatas["sitemap"].value_counts().rename("candidatas por sitemap")

### 2.3 Descarga de cada noticia
De cada página se extraen:

- `titular`: el `<h1>`; si falta, `og:title`.
- `bajada`: el `<h2 itemprop="name">` del encabezado; si falta (por un cambio de HTML), la meta descripción sin el titular repetido.
- `fecha_publicacion`: la meta `article:published_time`.

Cada respuesta se agrega a `datos/raw/articulos.jsonl` junto con el código HTTP, la URL final y la hora de descarga. Si el notebook se interrumpe o se reejecuta, continúa desde donde quedó y no repite solicitudes.

In [ ]:
RESPALDO = RAW / "articulos.jsonl"
ya = {}
if RESPALDO.exists():
    for linea in RESPALDO.read_text(encoding="utf-8").splitlines():
        d = json.loads(linea)
        ya[d["url"]] = d


def meta(soup, **attrs):
    tag = soup.find("meta", attrs=attrs)
    return tag["content"].strip() if tag and tag.get("content") else ""


def extraer(html):
    soup = BeautifulSoup(html, "lxml")
    h1 = soup.find("h1")
    titular = h1.get_text(" ", strip=True) if h1 else meta(soup, property="og:title")
    h2 = soup.find("h2", attrs={"itemprop": "name"})
    bajada = h2.get_text(" ", strip=True) if h2 else ""
    if not bajada:
        desc = meta(soup, name="description") or meta(soup, property="og:description")
        bajada = desc.removeprefix(titular).strip(" .|-")
    return {"titular": titular, "bajada": bajada,
            "fecha_publicacion": meta(soup, property="article:published_time"),
            "seccion_meta": meta(soup, property="article:section")}


pendientes = [u for u in candidatas["url"] if u not in ya]
print(f"Ya en respaldo: {len(ya)} | por descargar: {len(pendientes)} "
      f"(~{len(pendientes) * PAUSA_SEG / 60:.0f} min con pausas de {PAUSA_SEG} s)")

t0 = time.time()
with RESPALDO.open("a", encoding="utf-8") as f:
    for i, url in enumerate(pendientes, 1):
        r = descargar(url)
        d = {"url": url,
             "status": r.status_code if r is not None else None,
             "url_final": r.url if r is not None else None,
             "descargado_en": pd.Timestamp.now(tz="America/Lima").isoformat(timespec="seconds")}
        if r is not None and r.status_code == 200:
            d.update(extraer(r.text))
        f.write(json.dumps(d, ensure_ascii=False) + "\n")
        f.flush()
        ya[url] = d
        if i % 50 == 0:
            print(f"  {i}/{len(pendientes)} descargadas ({time.time() - t0:.0f} s)")
print(f"Descarga terminada en {time.time() - t0:.0f} s")

In [ ]:
df_raw = candidatas.merge(pd.DataFrame(list(ya.values())), on="url", how="left")
df_raw["status"] = df_raw["status"].astype("Int64")
df_raw.to_csv(RAW / "noticias_crudas.csv", index=False, encoding="utf-8")

horas = pd.to_datetime(df_raw["descargado_en"], format="ISO8601")
print("Filas crudas:", df_raw.shape)
print("Respuestas por código HTTP:", df_raw["status"].value_counts(dropna=False).to_dict())
print(f"Descargado entre {horas.min():%Y-%m-%d %H:%M} y {horas.max():%Y-%m-%d %H:%M} (hora de Lima)")
df_raw[["sitemap", "seccion_original", "status", "titular", "bajada"]].head()